# Commission Core Calc

In [ ]:
from load_wi_modules import *

In [ ]:
#- reload
fd = mergedDataFolder
fp = fd+'merged data.xlsx'
df = pd.read_excel(fp, dtype=dt)
da = df

df.Customer = df.Customer.fillna('--')

df.shape

In [ ]:
#- merge agents
agents.Rep = agents.Rep.str.title()
df = da.merge(agents, on='Rep', how='left')
df.shape

In [ ]:
ag = agents.set_index('Rep')
passthru = ag.Passthru.dropna()
passthru

## Act Comp

In [ ]:
## PAYMENT RULE
stats = ['Paid Act', 'Deact-React']
df['PayRep'] = df.PaidStatus.isin(stats)

idx = (df.AgentType=='Rep') & df.PaidStatus.str.contains('Slated')
df.loc[idx, 'PayRep'] = True

idx = (df.AgentType=='Dealer') & df.in_tmo
df.loc[idx, 'PayRep'] = True

#- payrep on house - don't wipe #s
idx = df.AgentType=='House'
df.loc[idx, 'PayRep'] = True

df.PayRep.value_counts()

In [ ]:
## Comp
idx = df.PayRep
df.loc[idx, 'perAgentComp'] = df.CompRate * df.LineMRC

df.perAgentComp = df.perAgentComp.round(2)
df.perAgentComp.value_counts().head()

In [ ]:
## Multipl thru
'''
There are neg qtys and per-line comps.
*.abs() keeps from negatives canceling.
'''
df['AgentComp'] = df.Qty.abs() * df.perAgentComp
df['Percentage'] = df.AgentComp / df.Incoming

df = df.round(2)

df.AgentComp.sum()

## Passthru

In [ ]:
passthru

In [ ]:
idx = df.Rep.isin(passthru.index)
df.loc[idx, 'Percentage'] = df.Rep.map(passthru)
df.loc[idx, 'AgentComp'] = df.Percentage * df.Incoming

cols = ['Rep', 'Incoming', 'Percentage', 'AgentComp',]
df.loc[idx, cols].head()

In [ ]:
#- rundown
gcols = ['AgentType', 'Rep', ]
df.groupby(gcols).AgentComp.sum().sort_index()

## Residual Comp

In [ ]:
residual.shape

In [ ]:
passthru

In [ ]:
#- merge agents
cols = ['AgentType', 'LastResidual', 'ResidualRate', 'Passthru']
ag = agents.set_index('Rep')[cols]
res = residual.merge(ag, on='Rep', how='left') 
res.shape    

In [ ]:
#- recode - not in agents table.
idx = ~res.Rep.isin(agents.Rep)
res.loc[idx, 'Rep'] = 'House-Alt'
res.Rep.value_counts().sort_index()

In [ ]:
#- recode - expired.
idx = res.PaymentNumber > res.LastResidual
res.loc[idx, 'Rep'] = 'House-Alt'
#res.Rep.value_counts().sort_index()

In [ ]:
idx = res.Rep.str.contains('House-Alt')
res.loc[idx, 'AgentType'] = 'House'

In [ ]:
#- agent comp
res['AgentComp'] = res.MRC * res.ResidualRate
res.AgentComp.sum()

In [ ]:
#- partner comp
idx = res.Passthru>0
res.loc[idx, 'AgentComp'] = res.TotalResidual * res.Passthru
res[idx].AgentComp.sum()

In [ ]:
resCompDetail = res.copy()

In [ ]:
#- roll up residual
gCols = ['Customer', 'BAN', 'Rep', 'AgentType', 'ActMonth', ]
cols = ['Qty', 'MRC', 'AgentComp', 'TotalResidual',]
g = resCompDetail.groupby(gCols)
res = g[cols].sum().reset_index()
res.shape

In [ ]:
#- percentage
idx = res.TotalResidual!=0
res.loc[idx, 'Percentage'] = res.AgentComp / res.TotalResidual

gc = ['Rep', 'Percentage']
res[gc].value_counts().sort_index()

In [ ]:
#- rundown
resComp = res.round(2)

idx = resComp.AgentComp > 0
gcols = ['AgentType', 'Rep', 'Percentage', ]
resComp[idx].groupby(gcols).AgentComp.sum().sort_index()

### Exports

In [ ]:
## act comp
fn = 'act comp.xlsx'
fp = mergedDataFolder+fn
df.to_excel(fp, index=False); print('Exported:', fn)

In [ ]:
## res comp
fn = 'res comp.xlsx'
fp = mergedDataFolder+fn
res.to_excel(fp, index=False); print('Exported:', fn)

### Review

**Rep Counts**

In [ ]:
repCounts

In [ ]:
gCols = ['Rep', 'ActivityType']
dx = df.groupby(gCols).AgentComp.sum()
dx[dx!=0]

**Paid Status**

In [ ]:
## Paid status rundown
idx = df.Big & ~df.IOT

#gCols = ['AgentType', 'Rep', ]
gCols = ['Rep', 'PaidStatus', 'Customer', ]
nCols = ['Qty', 'AgentComp']
s = df[idx].groupby(gCols)[nCols].sum()

fn = 'comp rundown - big orders.xlsx'
s.to_excel(mergedDataFolder+fn); print('Exported:', fn)
s